# 🤖 Bemo Chatbot — Gemini 2.5 Flash

> **Bemo** is a sharp, multilingual AI companion — built to feel genuinely human.
> She converses naturally in **Arabic, English, French, Franco (Arabizi), and any mix**,
> searches the web, solves math, checks weather,
> and reads **PDFs, PowerPoints, Word docs, Excel sheets, and images**.

---

| Feature | Detail |
|---|---|
| 🧠 Model | `gemini-2.5-flash` |
| 🔎 Web Search | DuckDuckGo |
| 🧮 Calculator | Built-in popup |
| 🌤️ Weather | wttr.in |
| 🕐 Date & Time | Python `datetime` |
| 📄 Files | PDF · DOCX · PPTX · XLSX · TXT · Images |
| 💬 Memory | Last 15 turns (full context) |
| 🎙️ Voice | TTS + STT |
| 📷 Camera | Live capture + vision |
| 🖥️ Interface | Tkinter dark-theme GUI |
| 🌐 Languages | Arabic · English · Français · Franco · Any mix |

In [45]:
!pip install -q \
    google-generativeai duckduckgo-search ddgs sympy \
    pyttsx3 SpeechRecognition pyaudio \
    opencv-python Pillow \
    pypdf python-docx python-pptx openpyxl

In [46]:
import os
from getpass import getpass

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("🔑  Enter your Google API key: ")

In [47]:
import google.generativeai as genai
import os, time, re

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
_raw_model = genai.GenerativeModel("gemini-2.5-flash")


class _ModelWrapper:
    """Wraps GenerativeModel — auto-retries on 429 with countdown."""

    def generate_content(self, prompt, **kwargs):
        max_attempts = 5
        wait = 20

        for attempt in range(1, max_attempts + 1):
            try:
                return _raw_model.generate_content(prompt, **kwargs)
            except Exception as e:
                msg = str(e)
                match = re.search(r"retry[^\d]*(\d+)", msg, re.I)
                suggested = int(match.group(1)) + 2 if match else wait

                if "429" in msg or "quota" in msg.lower():
                    if attempt == max_attempts:
                        raise
                    for s in range(suggested, 0, -1):
                        try:
                            status_var.set(
                                f"⏳ Rate limit — retrying in {s}s "
                                f"(attempt {attempt}/{max_attempts - 1})…"
                            )
                            root.update_idletasks()
                        except Exception:
                            pass
                        time.sleep(1)
                    wait = min(wait * 2, 120)
                else:
                    raise


model = _ModelWrapper()
print("✅ Model loaded — Gemini 2.5 Flash ready")


✅ Model loaded — Gemini 2.5 Flash ready


In [48]:
# ── 🔎 Web Search ──────────────────────────────────────────────────────────
from ddgs import DDGS

def web_search(query):
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=4):
            results.append(r["body"])
    return "\n".join(results) if results else "No results found."


# ── 🧮 Calculator ──────────────────────────────────────────────────────────
import sympy as sp

def calculator(query):
    expr = re.sub(r"[^0-9+\-*/().^ %]", " ", query).strip()
    try:
        return str(sp.sympify(expr))
    except Exception:
        return "Error: couldn't parse the expression."


# ── 🌤️ Weather ─────────────────────────────────────────────────────────────
import requests

def get_weather(city_query):
    m = re.search(
        r"(?:weather|طقس|جو|ta2s|t2s|gaw|clima|7arara)\s+(?:in|في|ف|fe|f)?\s*(\w+)",
        city_query, re.I,
    )
    city = m.group(1) if m else city_query.split()[-1]
    try:
        return requests.get(f"https://wttr.in/{city}?format=3", timeout=5).text
    except Exception:
        return "Weather service unavailable."


# ── 🕐 Date & Time ─────────────────────────────────────────────────────────
from datetime import datetime

def get_datetime(_=None):
    now = datetime.now()
    return now.strftime("📅 %A, %d %B %Y  |  🕐 %I:%M %p")

In [49]:
MAX_MEMORY = 15      # keep last N turns
memory     = []      # list of {"user": ..., "bot": ...}

# last uploaded file kept so user can ask follow-up questions
_last_file_context = {"name": None, "content": None}


def save_memory(user, bot):
    memory.append({"user": user, "bot": bot})
    if len(memory) > MAX_MEMORY:
        memory.pop(0)


def get_memory():
    if not memory:
        return ""
    lines = []
    for turn in memory:
        lines.append(f"User: {turn['user']}")
        lines.append(f"Bemo: {turn['bot']}")
    return "\n".join(lines)


def memory_count():
    return len(memory)


In [50]:
import os
import PIL.Image

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".gif", ".webp", ".bmp"}


def _read_pdf(path):
    from pypdf import PdfReader
    reader = PdfReader(path)
    text, char_budget = [], 12_000
    for page in reader.pages:
        chunk = page.extract_text() or ""
        text.append(chunk)
        char_budget -= len(chunk)
        if char_budget <= 0:
            break
    return "\n".join(text)[:12_000]


def _read_docx(path):
    import docx
    doc = docx.Document(path)
    parts = []
    for para in doc.paragraphs:
        if para.text.strip():
            parts.append(para.text)
    # also pull table cells
    for table in doc.tables:
        for row in table.rows:
            parts.append("\t".join(c.text for c in row.cells))
    return "\n".join(parts)[:12_000]


def _read_pptx(path):
    from pptx import Presentation
    prs = Presentation(path)
    parts = []
    for i, slide in enumerate(prs.slides, 1):
        parts.append(f"\n--- Slide {i} ---")
        for shape in slide.shapes:
            if hasattr(shape, "text") and shape.text.strip():
                parts.append(shape.text.strip())
    return "\n".join(parts)[:12_000]


def _read_xlsx(path):
    import openpyxl
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    parts = []
    for sheet_name in wb.sheetnames:
        ws = wb[sheet_name]
        parts.append(f"\n--- Sheet: {sheet_name} ---")
        for row in ws.iter_rows(max_row=80, values_only=True):
            row_str = "\t".join(str(c) if c is not None else "" for c in row)
            if row_str.strip():
                parts.append(row_str)
    return "\n".join(parts)[:12_000]


def read_file_content(path):
    """Return (text_content, is_image). For images returns (None, True)."""
    ext = os.path.splitext(path)[1].lower()
    try:
        if ext == ".pdf":
            return _read_pdf(path), False
        elif ext == ".docx":
            return _read_docx(path), False
        elif ext == ".pptx":
            return _read_pptx(path), False
        elif ext in (".xlsx", ".xls", ".xlsm"):
            return _read_xlsx(path), False
        elif ext in IMAGE_EXTS:
            return None, True
        else:
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                return f.read()[:12_000], False
    except Exception as e:
        return f"[Error reading file: {e}]", False


def summarize_file(path, question=None):
    """Summarize any supported file. Stores content for follow-up questions."""
    fname = os.path.basename(path)
    ext   = os.path.splitext(path)[1].lower()

    content, is_image = read_file_content(path)

    if is_image:
        try:
            img = PIL.Image.open(path)
            q   = question or "Describe this image in detail. What do you see?"
            res = _raw_model.generate_content([q, img])
            ans = res.text
        except Exception as e:
            ans = f"Could not process image: {e}"
        _last_file_context["name"]    = fname
        _last_file_context["content"] = f"[Image: {fname}] — Vision description above."
        return ans

    if not content or content.startswith("[Error"):
        return content or "Could not read file."

    _last_file_context["name"]    = fname
    _last_file_context["content"] = content

    file_type_labels = {
        ".pdf":  "PDF document",
        ".docx": "Word document",
        ".pptx": "PowerPoint presentation",
        ".xlsx": "Excel spreadsheet",
        ".xls":  "Excel spreadsheet",
        ".xlsm": "Excel spreadsheet",
    }
    label = file_type_labels.get(ext, "file")

    prompt = (
        "You are Bemo 🤖 — a helpful AI assistant.\n"
        f"The user uploaded a {label} named \"{fname}\".\n\n"
        "Your tasks:\n"
        "1. Give a clear, structured summary (use bullet points or sections as needed).\n"
        "2. Highlight the most important points.\n"
        "3. End with 2-3 follow-up questions the user might want to ask about this file.\n\n"
        "Respond in the same language as the document content.\n\n"
        f"File content:\n{content}"
    )
    res = model.generate_content(prompt)
    return res.text


print("✅ File reader ready — supports PDF · DOCX · PPTX · XLSX · TXT · Images")


✅ File reader ready — supports PDF · DOCX · PPTX · XLSX · TXT · Images


In [51]:
WEATHER_KW  = {
    "weather", "طقس", "جو", "حرارة", "درجة", "temperature", "forecast",
    # Franco Arabic
    "ta2s", "t2s", "gaw", "7arara", "clima",
}
DATETIME_KW = {
    "time", "date", "وقت", "تاريخ", "النهارده", "today", "اليوم",
    "الساعة", "now", "clock", "day", "month", "year",
    # Franco Arabic
    "sa3a", "yom", "ennaharda", "el-yom", "elsa3a",
}
CALC_KW     = {
    "calc", "calculate", "حساب", "احسب", "يساوي", "equals", "compute",
    # Franco Arabic
    "e7seb", "7esab", "yesawi",
}
TOOLS       = {"SEARCH", "CALCULATE", "WEATHER", "DATETIME", "NONE"}


def fast_decide(text):
    low   = text.lower()
    words = set(low.split())

    if words & WEATHER_KW:
        return "WEATHER"
    if words & DATETIME_KW:
        return "DATETIME"
    if words & CALC_KW:
        return "CALCULATE"
    math_chars = sum(1 for c in text if c in "0123456789+-*/().^ ")
    if math_chars / max(len(text), 1) > 0.55:
        return "CALCULATE"
    return None


def decide_tool(user_input):
    quick = fast_decide(user_input)
    if quick:
        return quick
    prompt = (
        "Reply with ONE word only — SEARCH, CALCULATE, WEATHER, DATETIME, or NONE.\n"
        "SEARCH only if real-time/factual web data is needed.\n"
        f"Question: {user_input}"
    )
    res  = model.generate_content(prompt)
    dec  = res.text.strip().upper()
    for t in TOOLS:
        if t in dec:
            return t
    return "NONE" 

In [52]:
PERSONALITY = """
You are Bemo 🤖 — sharp, warm, multilingual, and genuinely helpful.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
WHO YOU ARE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
You are not a corporate bot. You are the kind of brilliant friend everyone
wishes they had — the one who gives real answers, real opinions, and real care.
You are curious, direct, occasionally witty, and always on the person's side.
You pick up on mood and adapt: supportive when someone is struggling, focused
when they need precision, playful when the moment is light.
You genuinely enjoy helping — it is not a task, it is who you are.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LANGUAGE — DETECT & MIRROR PERFECTLY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Detect the user's language from their very first message and respond in it —
no asking, no switching unless they do.

• Arabic (فصحى or any dialect) → respond naturally in Arabic
• English → respond in English
• French → respond in French
• Franco Arabic / Arabizi → respond in Franco naturally, same warmth
  Common patterns: "3ayez", "msh 3arif", "kol 7aga", "bs", "ya3ni",
  "mesh lazem", "walla", "7aga", "e7ki", "3la ra7tak", "t3ala", "mashy"
• Code-switching / mixed → mirror the exact mix — do not normalize it
• Any other language → detect and respond in it fluently

Never ask "what language do you prefer?" — just detect and respond.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESPONSE STYLE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Default to concise — say exactly what needs to be said, nothing more
• Go deep only when the topic genuinely demands depth
• Use markdown naturally: **bold** for key terms, bullet lists for enumerations,
  `code blocks` for code
• Dry, light humor when the moment is right — never forced, never cringe
• If something is unclear, ask ONE sharp question — not five vague ones
• When re-explaining, use a different angle — never repeat louder

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
HARD RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✗  Never say "As an AI…", "I cannot feel…", or "I don't have opinions…"
✗  Never be condescending or patronizing
✗  Never pad responses with filler like "Great question!" or "Certainly!"
✓  If a file was uploaded, remember it fully — answer any follow-up about it
✓  Reference conversation history naturally — feel like a continuous presence
✓  Be direct. Say what you actually think. Have a point of view.
✓  If you disagree with something, say so — respectfully but honestly
"""


def _file_context_section():
    if _last_file_context["content"]:
        return (
            f"\n\n[Uploaded file in context: {_last_file_context['name']}]\n"
            f"{_last_file_context['content'][:3000]}\n[end of file excerpt]"
        )
    return ""


def generate_response(user_input, tool, tool_result):
    history      = get_memory()
    history_sec  = f"Conversation history:\n{history}\n" if history else ""
    tool_sec     = f"\nTool result:\n{tool_result}" if tool_result else ""
    file_sec     = _file_context_section()

    prompt = f"""{PERSONALITY}

{history_sec}{file_sec}

User: {user_input}
{tool_sec}

Reply naturally. Reference the conversation or file content when relevant.
"""
    res = model.generate_content(prompt)
    return res.text


import threading

TOOL_FN = {
    "SEARCH":    web_search,
    "CALCULATE": calculator,
    "WEATHER":   get_weather,
    "DATETIME":  get_datetime,
}


def agent(user_input):
    tool        = decide_tool(user_input)
    tool_result = TOOL_FN[tool](user_input) if tool in TOOL_FN else None
    response    = generate_response(user_input, tool, tool_result)
    save_memory(user_input, response)
    return response


print("✅ Agent ready")

✅ Agent ready


In [53]:
import pyttsx3, threading
import speech_recognition as sr

_tts_engine = pyttsx3.init()
_tts_engine.setProperty("rate", 162)
_tts_engine.setProperty("volume", 1.0)
_tts_lock   = threading.Lock()
tts_enabled = True


def speak(text):
    if not tts_enabled:
        return
    clean = re.sub(r"[*_`#>]", "", text)   # strip markdown
    def _run():
        with _tts_lock:
            _tts_engine.say(clean[:600])
            _tts_engine.runAndWait()
    threading.Thread(target=_run, daemon=True).start()


def listen_once(on_result, on_error, on_status):
    def _listen():
        recognizer = sr.Recognizer()
        recognizer.energy_threshold = 300
        recognizer.dynamic_energy_threshold = True
        with sr.Microphone() as source:
            on_status("🎤 Adjusting for noise…")
            recognizer.adjust_for_ambient_noise(source, duration=0.8)
            on_status("🎤 Listening… speak now")
            try:
                audio = recognizer.listen(source, timeout=8, phrase_time_limit=15)
                on_status("🔄 Recognizing…")
                text = recognizer.recognize_google(audio)
                on_result(text)
            except sr.WaitTimeoutError:
                on_error("No speech detected — try again")
            except sr.UnknownValueError:
                on_error("Couldn't understand — try again")
            except Exception as e:
                on_error(str(e))
    threading.Thread(target=_listen, daemon=True).start()


In [54]:
import cv2, tempfile, tkinter as tk
from PIL import Image, ImageTk


def capture_from_camera(on_captured, on_error):
    """Opens a Toplevel window showing webcam feed; user presses Space to capture."""
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        on_error("No camera found")
        return

    win = tk.Toplevel()
    win.title("📷  Press Space to capture — Esc to cancel")
    win.configure(bg="#1e1e2e")
    lbl = tk.Label(win, bg="#1e1e2e")
    lbl.pack(padx=10, pady=10)
    tk.Label(win, text="Space = capture   |   Esc = cancel",
             font=("Segoe UI", 10), bg="#1e1e2e", fg="#cdd6f4").pack(pady=(0, 8))

    _running = [True]

    def update():
        if not _running[0]:
            return
        ret, frame = cap.read()
        if ret:
            rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img   = Image.fromarray(rgb).resize((560, 420))
            photo = ImageTk.PhotoImage(img)
            lbl.configure(image=photo)
            lbl.image = photo
        win.after(30, update)

    def capture(event=None):
        _running[0] = False
        ret, frame  = cap.read()
        cap.release()
        win.destroy()
        if ret:
            tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
            cv2.imwrite(tmp.name, frame)
            on_captured(tmp.name)
        else:
            on_error("Capture failed")

    def cancel(event=None):
        _running[0] = False
        cap.release()
        win.destroy()

    win.bind("<space>",  capture)
    win.bind("<Escape>", cancel)
    win.focus_set()
    update()


In [55]:
import tkinter as tk
from tkinter import filedialog
import threading, re, os

# ── Colour Palette (Catppuccin Mocha — refined & softened) ─────────────────
BASE        = "#1e1e2e"
MANTLE      = "#181825"
CRUST       = "#11111b"
SURFACE0    = "#313244"
SURFACE1    = "#45475a"
SURFACE2    = "#585b70"
OVERLAY0    = "#6c7086"
OVERLAY1    = "#7f849c"
TEXT_C      = "#cdd6f4"
SUBTEXT1    = "#bac2de"
SUBTEXT0    = "#a6adc8"
BLUE        = "#89b4fa"
LAVENDER    = "#b4befe"
MAUVE       = "#cba6f7"
RED_C       = "#f38ba8"
PEACH       = "#fab387"
YELLOW_C    = "#f9e2af"
GREEN_C     = "#a6e3a1"
TEAL        = "#94e2d5"

# Semantic aliases — slightly warmed up bubble colours
BG          = BASE
BUBBLE_L    = "#2a2a3d"      # softer, slightly warmer left bubble
BUBBLE_R    = "#353550"      # softer, slightly warmer right bubble
TEXT_CLR    = TEXT_C
ACCENT      = "#8bb8fa"      # slightly softer blue accent
INPUT_BG    = SURFACE0
BTN_BG      = "#7aadf5"     # slightly richer send button
BTN_FG      = CRUST
STATUS_CLR  = OVERLAY1
BOT_BORDER  = LAVENDER
USER_BORDER = TEAL
SUBTEXT     = SUBTEXT0
GREEN       = GREEN_C
RED         = RED_C
YELLOW      = YELLOW_C

FONT_CHAT   = ("Segoe UI", 11)
FONT_UI     = ("Segoe UI", 10)
FONT_BOLD   = ("Segoe UI", 11, "bold")
FONT_SMALL  = ("Segoe UI", 9)
FONT_MONO   = ("Consolas", 10)

FILE_TYPES = [
    ("All supported files",
     "*.pdf *.docx *.pptx *.xlsx *.xls *.xlsm *.txt *.md *.py *.csv "
     "*.jpg *.jpeg *.png *.gif *.webp *.bmp"),
    ("PDF",              "*.pdf"),
    ("Word Document",    "*.docx"),
    ("PowerPoint",       "*.pptx"),
    ("Excel",            "*.xlsx *.xls *.xlsm"),
    ("Text / Code",      "*.txt *.md *.py *.csv *.log"),
    ("Images",           "*.jpg *.jpeg *.png *.gif *.webp *.bmp"),
    ("All files",        "*.*"),
]

FILE_ICONS = {
    ".pdf":  "📕", ".docx": "📘", ".pptx": "📙",
    ".xlsx": "📗", ".xls":  "📗", ".xlsm": "📗",
    ".jpg":  "🖼️", ".jpeg": "🖼️", ".png": "🖼️",
    ".gif":  "🖼️", ".webp": "🖼️", ".bmp": "🖼️",
    ".txt":  "📄", ".md":   "📄", ".py":  "🐍",
    ".csv":  "📊",
}


# ── Hover helper ───────────────────────────────────────────────────────────
def add_hover(widget, hover_bg, normal_bg=None):
    if normal_bg is None:
        normal_bg = widget.cget("bg")
    widget.bind("<Enter>", lambda e: widget.config(bg=hover_bg))
    widget.bind("<Leave>", lambda e: widget.config(bg=normal_bg))


# ── Rounded-frame helper (fake rounded corners via nested frames) ──────────
def make_rounded_frame(parent, bg, pad=6, inner_bg=None):
    """Creates a padded frame that simulates softer rounded edges."""
    if inner_bg is None:
        inner_bg = bg
    outer = tk.Frame(parent, bg=bg, padx=pad, pady=pad)
    inner = tk.Frame(outer, bg=inner_bg)
    inner.pack(fill=tk.BOTH, expand=True)
    return outer, inner


# ── Markdown-lite renderer ─────────────────────────────────────────────────
def insert_markdown(widget, text, base_tag):
    widget.configure(state="normal")
    pattern = re.compile(r"(\*\*(.+?)\*\*|\*(.+?)\*|`(.+?)`)", re.S)
    last = 0
    for m in pattern.finditer(text):
        widget.insert(tk.END, text[last:m.start()], (base_tag,))
        full = m.group(0)
        if full.startswith("**"):
            widget.insert(tk.END, m.group(2), (base_tag, "bold"))
        elif full.startswith("*"):
            widget.insert(tk.END, m.group(3), (base_tag, "italic"))
        else:
            widget.insert(tk.END, m.group(4), (base_tag, "code"))
        last = m.end()
    widget.insert(tk.END, text[last:], (base_tag,))
    widget.configure(state="disabled")


# ── Chat area ──────────────────────────────────────────────────────────────
def make_chat_area(parent):
    frame = tk.Frame(parent, bg=BG)
    frame.pack(fill=tk.BOTH, expand=True)

    canvas = tk.Canvas(frame, bg=BG, highlightthickness=0)
    vsb = tk.Scrollbar(frame, orient="vertical", command=canvas.yview,
                       bg=SURFACE1, troughcolor=BG, activebackground=ACCENT,
                       width=6, relief="flat", bd=0)
    canvas.configure(yscrollcommand=vsb.set)
    vsb.pack(side=tk.RIGHT, fill=tk.Y)
    canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

    inner = tk.Frame(canvas, bg=BG)
    inner_id = canvas.create_window((0, 0), window=inner, anchor="nw")

    def _on_resize(e):
        canvas.itemconfig(inner_id, width=e.width)
    canvas.bind("<Configure>", _on_resize)

    def _on_inner_configure(e):
        canvas.configure(scrollregion=canvas.bbox("all"))
    inner.bind("<Configure>", _on_inner_configure)

    def _scroll(e):
        canvas.yview_scroll(int(-1 * (e.delta / 120)), "units")
    canvas.bind_all("<MouseWheel>", _scroll)

    return inner, canvas


def add_bubble(inner, canvas, text, side, file_label=None):
    """Chat bubble with avatar indicator, accent border, and softer appearance."""
    is_left = (side == "left")

    # Increased vertical spacing for WhatsApp-like breathing room
    row = tk.Frame(inner, bg=BG)
    row.pack(fill=tk.X, padx=14, pady=(6, 3))

    av_text = "B" if is_left else "U"
    av_bg   = LAVENDER if is_left else TEAL

    # Softer rounded avatar with more padding
    avatar = tk.Label(row, text=av_text, font=("Segoe UI", 9, "bold"),
                      bg=av_bg, fg=CRUST, width=2, padx=5, pady=5)

    if file_label:
        fl_row = tk.Frame(inner, bg=BG)
        fl_row.pack(fill=tk.X, padx=62, pady=(0, 3))
        tk.Label(fl_row, text=file_label, font=FONT_SMALL,
                 bg=BG, fg=ACCENT).pack(side=tk.LEFT if is_left else tk.RIGHT)

    bubble_bg = BUBBLE_L if is_left else BUBBLE_R
    tag       = "left_text" if is_left else "right_text"

    if is_left:
        avatar.pack(side=tk.LEFT, anchor="n", padx=(2, 0), pady=(5, 0))
        # Slightly thicker, softer accent border
        tk.Frame(row, bg=BOT_BORDER, width=3).pack(
            side=tk.LEFT, fill=tk.Y, padx=(5, 7))

    # Bubble wrapper for softer edge simulation
    bubble_wrap = tk.Frame(row, bg=bubble_bg, padx=2, pady=2)

    bubble = tk.Text(
        bubble_wrap, wrap=tk.WORD, bg=bubble_bg, fg=TEXT_CLR,
        relief="flat", font=FONT_CHAT, padx=16, pady=12,
        bd=0, highlightthickness=0, cursor="arrow",
        spacing1=2, spacing3=2,  # line breathing room
    )
    bubble.tag_configure("bold",   font=(FONT_CHAT[0], FONT_CHAT[1], "bold"))
    bubble.tag_configure("italic", font=(FONT_CHAT[0], FONT_CHAT[1], "italic"))
    bubble.tag_configure("code",   font=FONT_MONO, background=CRUST, foreground=GREEN)
    bubble.tag_configure("left_text",  foreground=TEXT_CLR, lmargin1=2, lmargin2=2)
    bubble.tag_configure("right_text", foreground=TEXT_CLR, lmargin1=2, lmargin2=2)

    insert_markdown(bubble, text, tag)

    lines   = text.count("\n") + 1
    wrapped = sum(max(1, len(l) // 60 + 1) for l in text.split("\n"))
    bubble.configure(height=min(max(wrapped, lines), 40))
    bubble.pack(fill=tk.BOTH, expand=True)

    if is_left:
        bubble_wrap.pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0, 56))
    else:
        bubble_wrap.pack(side=tk.RIGHT, fill=tk.X, expand=True, padx=(56, 0))
        tk.Frame(row, bg=USER_BORDER, width=3).pack(
            side=tk.RIGHT, fill=tk.Y, padx=(7, 5))
        avatar.pack(side=tk.RIGHT, anchor="n", padx=(0, 2), pady=(5, 0))

    canvas.update_idletasks()
    canvas.yview_moveto(1.0)


def add_typing_indicator(inner, canvas):
    frame = tk.Frame(inner, bg=BG)
    frame.pack(fill=tk.X, padx=(58, 64), pady=(6, 3))
    pill = tk.Frame(frame, bg=BUBBLE_L, padx=20, pady=11)
    pill.pack(side=tk.LEFT)
    tk.Label(pill, text="● ● ●", font=("Segoe UI", 10),
             bg=BUBBLE_L, fg=OVERLAY0).pack()
    canvas.update_idletasks()
    canvas.yview_moveto(1.0)
    return frame


# ── Calculator popup ───────────────────────────────────────────────────────
def open_calculator():
    win = tk.Toplevel()
    win.title("🧮  Calculator")
    win.resizable(False, False)
    win.configure(bg=MANTLE)

    expr = tk.StringVar(value="")
    display = tk.Entry(
        win, textvariable=expr,
        font=("Segoe UI", 20, "bold"),
        bg=CRUST, fg=TEXT_C,
        insertbackground=ACCENT,
        relief="flat", justify="right", bd=16,
    )
    display.grid(row=0, column=0, columnspan=4, sticky="ew", padx=14, pady=14)

    def press(val):
        if val == "=":
            try:   expr.set(str(eval(expr.get())))
            except: expr.set("Error")
        elif val == "C":  expr.set("")
        elif val == "⌫": expr.set(expr.get()[:-1])
        else:             expr.set(expr.get() + str(val))

    BTN_COLORS = {
        "=": (BLUE, CRUST), "C": (RED_C, CRUST), "⌫": (PEACH, CRUST),
        "/": (MAUVE, CRUST), "*": (MAUVE, CRUST),
        "+": (MAUVE, CRUST), "-": (MAUVE, CRUST), "%": (MAUVE, CRUST),
    }
    buttons = [
        ["C",  "⌫", "%",  "/"],
        ["7",  "8", "9",  "*"],
        ["4",  "5", "6",  "-"],
        ["1",  "2", "3",  "+"],
        ["00", "0", ".",  "="],
    ]
    for r, row_btns in enumerate(buttons, 1):
        for c, val in enumerate(row_btns):
            bg, fg = BTN_COLORS.get(val, (SURFACE0, TEXT_C))
            btn = tk.Button(win, text=val, font=("Segoe UI", 13, "bold"),
                            bg=bg, fg=fg,
                            activebackground=TEXT_C, activeforeground=bg,
                            relief="flat", width=5, height=2, cursor="hand2",
                            command=lambda v=val: press(v))
            btn.grid(row=r, column=c, padx=4, pady=4)
            add_hover(btn, OVERLAY0 if val not in BTN_COLORS else SUBTEXT1, bg)

    win.bind("<Return>",    lambda e: press("="))
    win.bind("<BackSpace>", lambda e: press("⌫"))
    win.bind("<Escape>",    lambda e: win.destroy())


# ── Main Window ────────────────────────────────────────────────────────────
root = tk.Tk()
root.title("Bemo — AI Companion")
root.geometry("900x720")
root.minsize(640, 500)
root.configure(bg=BG)

status_var = tk.StringVar(value="Ready ✨")

# ── Header ─────────────────────────────────────────────────────────────────
header = tk.Frame(root, bg=MANTLE)
header.pack(fill=tk.X)

header_inner = tk.Frame(header, bg=MANTLE, pady=14)
header_inner.pack(fill=tk.X, padx=22)

logo_frame = tk.Frame(header_inner, bg=MANTLE)
logo_frame.pack(side=tk.LEFT)
tk.Label(logo_frame, text="🤖", font=("Segoe UI", 22),
         bg=MANTLE, fg=TEXT_C).pack(side=tk.LEFT, padx=(0, 12))
name_col = tk.Frame(logo_frame, bg=MANTLE)
name_col.pack(side=tk.LEFT)
tk.Label(name_col, text="Bemo",
         font=("Segoe UI", 16, "bold"), bg=MANTLE, fg=ACCENT).pack(anchor="w")

# Subtle online indicator dot + subtitle
subtitle_frame = tk.Frame(name_col, bg=MANTLE)
subtitle_frame.pack(anchor="w")
tk.Label(subtitle_frame, text="●", font=("Segoe UI", 7),
         bg=MANTLE, fg=GREEN_C).pack(side=tk.LEFT, padx=(0, 5))
tk.Label(subtitle_frame, text="Online  ·  Arabic · English · Français · Franco",
         font=FONT_SMALL, bg=MANTLE, fg=OVERLAY1).pack(side=tk.LEFT)


def _make_hbtn(parent, text, fg, cmd, text_var=None):
    b = tk.Button(parent, text=text, font=FONT_UI,
                  bg=SURFACE0, fg=fg, relief="flat",
                  padx=14, pady=6, cursor="hand2", bd=0,
                  activebackground=SURFACE1, activeforeground=fg,
                  command=cmd)
    if text_var:
        b.config(textvariable=text_var)
    add_hover(b, SURFACE1, SURFACE0)
    return b


def clear_chat():
    for w in chat_inner.winfo_children():
        w.destroy()
    memory.clear()
    _last_file_context["name"]    = None
    _last_file_context["content"] = None
    _show_welcome()
    status_var.set("Chat cleared  ✨")

_tts_var = tk.StringVar(value="🔊  Sound ON")
def toggle_tts():
    global tts_enabled
    tts_enabled = not tts_enabled
    _tts_var.set("🔊  Sound ON" if tts_enabled else "🔇  Sound OFF")

_make_hbtn(header_inner, "🗑  Clear",  RED_C, clear_chat).pack(side=tk.RIGHT, padx=(5, 0))
_make_hbtn(header_inner, "🧮  Calc",   PEACH, open_calculator).pack(side=tk.RIGHT, padx=(5, 0))
_make_hbtn(header_inner, "", ACCENT, toggle_tts, _tts_var).pack(side=tk.RIGHT, padx=(5, 0))

# Softer gradient-like accent divider under header
accent_div = tk.Frame(root, bg=ACCENT, height=2)
accent_div.pack(fill=tk.X)
# Secondary subtle shadow line
tk.Frame(root, bg="#15152a", height=1).pack(fill=tk.X)

# Chat area
chat_inner, chat_canvas = make_chat_area(root)

# Softer separator above input
tk.Frame(root, bg="#252540", height=1).pack(fill=tk.X)

# ── Input row ──────────────────────────────────────────────────────────────
input_row = tk.Frame(root, bg=MANTLE, pady=12)
input_row.pack(fill=tk.X, padx=16)

def set_status(msg):
    status_var.set(msg)


# ── Send ───────────────────────────────────────────────────────────────────
def send_message(event=None):
    text = user_input.get().strip()
    if not text:
        return
    user_input.delete(0, tk.END)
    add_bubble(chat_inner, chat_canvas, text, "right")
    send_btn.config(state="disabled")
    set_status("Bemo is thinking…")
    typing_frame = add_typing_indicator(chat_inner, chat_canvas)

    def _run():
        file_kws = {
            "explain", "summarize", "what", "tell", "describe",
            "اشرح", "لخص", "ايه", "ما", "وضح",
            "e7ki", "e2ra", "wad7li", "shoof", "2olly",
        }
        has_file  = _last_file_context["content"] is not None
        is_file_q = has_file and bool(set(text.lower().split()) & file_kws)

        if is_file_q:
            resp = generate_response(text, None, None)
            save_memory(text, resp)
        else:
            resp = agent(text)

        def _update():
            try: typing_frame.destroy()
            except: pass
            add_bubble(chat_inner, chat_canvas, resp, "left")
            send_btn.config(state="normal")
            set_status(f"✅  Ready  ·  {memory_count()} turns in memory")
            threading.Thread(target=speak, args=(resp,), daemon=True).start()
        root.after(0, _update)

    threading.Thread(target=_run, daemon=True).start()


# ── Upload ─────────────────────────────────────────────────────────────────
def _handle_file_path(path, fname):
    ext  = os.path.splitext(fname)[1].lower()
    icon = FILE_ICONS.get(ext, "📎")
    set_status(f"📂  Reading {fname}…")
    typing_frame = add_typing_indicator(chat_inner, chat_canvas)

    def _run():
        summary = summarize_file(path)
        def _update():
            try: typing_frame.destroy()
            except: pass
            add_bubble(chat_inner, chat_canvas, summary, "left",
                       file_label=f"{icon}  {fname}")
            send_btn.config(state="normal")
            upload_btn.config(state="normal")
            set_status(f"✅  {fname} ready  ·  Ask me anything about it")
            threading.Thread(target=speak, args=(summary[:400],), daemon=True).start()
        root.after(0, _update)

    threading.Thread(target=_run, daemon=True).start()


def upload_file():
    path = filedialog.askopenfilename(
        title="Choose a file for Bemo to analyze",
        filetypes=FILE_TYPES,
    )
    if not path:
        return
    fname = os.path.basename(path)
    ext   = os.path.splitext(fname)[1].lower()
    icon  = FILE_ICONS.get(ext, "📎")
    upload_btn.config(state="disabled")
    send_btn.config(state="disabled")
    add_bubble(chat_inner, chat_canvas,
               f"{icon}  Uploaded: **{fname}**", "right")
    _handle_file_path(path, fname)


# ── Image / Camera ─────────────────────────────────────────────────────────
def _handle_image_path(path, fname, question=None):
    set_status(f"🖼️  Processing {fname}…")
    typing_frame = add_typing_indicator(chat_inner, chat_canvas)

    def _run():
        try:
            img = PIL.Image.open(path)
            q   = question or "Describe what you see in this image in detail."
            res = _raw_model.generate_content([q, img])
            ans = res.text
        except Exception as e:
            ans = f"Could not process image: {e}"

        _last_file_context["name"]    = fname
        _last_file_context["content"] = f"[Image analyzed: {fname}]"
        save_memory(f"[User sent image: {fname}] {question or ''}", ans)

        def _update():
            try: typing_frame.destroy()
            except: pass
            add_bubble(chat_inner, chat_canvas, ans, "left",
                       file_label=f"🖼️  {fname}")
            send_btn.config(state="normal")
            cam_btn.config(state="normal")
            set_status("✅  Image analyzed")
            threading.Thread(target=speak, args=(ans[:400],), daemon=True).start()
        root.after(0, _update)

    threading.Thread(target=_run, daemon=True).start()


def open_camera():
    cam_btn.config(state="disabled")
    send_btn.config(state="disabled")

    def on_captured(img_path):
        q = user_input.get().strip() or "What do you see in this image?"
        user_input.delete(0, tk.END)
        add_bubble(chat_inner, chat_canvas, f"📷  Camera — {q}", "right")
        root.after(0, lambda: _handle_image_path(img_path, "camera_capture.png", q))

    def on_error(msg):
        root.after(0, lambda: set_status(f"❌  Camera: {msg}"))
        root.after(0, lambda: cam_btn.config(state="normal"))
        root.after(0, lambda: send_btn.config(state="normal"))

    root.after(0, lambda: capture_from_camera(on_captured, on_error))


# ── Voice ──────────────────────────────────────────────────────────────────
def start_listening():
    mic_btn.config(state="disabled")
    send_btn.config(state="disabled")

    def on_result(text):
        def _insert():
            user_input.delete(0, tk.END)
            user_input.insert(0, text)
            mic_btn.config(state="normal")
            send_btn.config(state="normal")
            set_status("🎤  Got it — press Send or Enter")
        root.after(0, _insert)

    def on_error(msg):
        root.after(0, lambda: set_status(f"❌  {msg}"))
        root.after(0, lambda: mic_btn.config(state="normal"))
        root.after(0, lambda: send_btn.config(state="normal"))

    listen_once(on_result, on_error, set_status)


# ── Input widgets ──────────────────────────────────────────────────────────
def _icon_btn(parent, text, fg, cmd):
    b = tk.Button(parent, text=text, font=("Segoe UI", 14),
                  bg=SURFACE0, fg=fg, relief="flat",
                  padx=11, pady=6, cursor="hand2", bd=0,
                  activebackground=SURFACE1, activeforeground=fg,
                  command=cmd)
    add_hover(b, SURFACE1, SURFACE0)
    return b

upload_btn = _icon_btn(input_row, "📎", ACCENT,  upload_file)
upload_btn.pack(side=tk.LEFT, padx=(0, 4))

mic_btn    = _icon_btn(input_row, "🎤", RED_C,   start_listening)
mic_btn.pack(side=tk.LEFT, padx=(0, 4))

cam_btn    = _icon_btn(input_row, "📷", GREEN_C, open_camera)
cam_btn.pack(side=tk.LEFT, padx=(0, 10))

# Input field with softer focus-ring border and more padding
input_border = tk.Frame(input_row, bg=SURFACE1, padx=2, pady=2)
input_border.pack(side=tk.LEFT, fill=tk.X, expand=True)

user_input = tk.Entry(
    input_border, font=FONT_CHAT,
    bg=INPUT_BG, fg=TEXT_CLR,
    insertbackground=ACCENT,
    relief="flat", bd=10,
)
user_input.pack(fill=tk.X)
user_input.bind("<Return>",   send_message)
user_input.bind("<FocusIn>",  lambda e: input_border.config(bg=ACCENT))
user_input.bind("<FocusOut>", lambda e: input_border.config(bg=SURFACE1))

send_btn = tk.Button(
    input_row, text="Send  ➤",
    font=("Segoe UI", 10, "bold"),
    bg=BTN_BG, fg=BTN_FG,
    relief="flat", padx=20, pady=9,
    cursor="hand2", bd=0,
    activebackground=LAVENDER, activeforeground=BTN_FG,
    command=send_message,
)
add_hover(send_btn, LAVENDER, BTN_BG)
send_btn.pack(side=tk.RIGHT, padx=(10, 0))

# ── Status bar ─────────────────────────────────────────────────────────────
tk.Frame(root, bg="#252540", height=1).pack(fill=tk.X)
status_bar = tk.Frame(root, bg=MANTLE)
status_bar.pack(fill=tk.X)
tk.Label(status_bar, textvariable=status_var, font=FONT_SMALL,
         bg=MANTLE, fg=STATUS_CLR, anchor="w", pady=6
         ).pack(fill=tk.X, padx=20)

# ── Welcome message ────────────────────────────────────────────────────────
def _show_welcome():
    add_bubble(
        chat_inner, chat_canvas,
        "Hi there! 👋  I'm **Bemo**, your AI assistant.\n\n"
        "I can chat in **Arabic**, **English**, **Français**, and **Franco** — "
        "feel free to mix however you like.\n\n"
        "Send me a message, upload a file, snap a photo, or just say hi.\n"
        "كلمني بأي لغة تحبها — أنا جاهز! 💬",
        "left",
    )

_show_welcome()
user_input.focus_set()
root.mainloop()